# Regresi Linear dan Pemisahan Data (Train-Test Split)

Course 2: Supervised Learning

Di notebook ini saya mencoba membuat model regresi linear untuk memprediksi harga rumah. Yang dibandingkan: data dengan dan tanpa one-hot encoding, lalu pengaruh scaling terhadap error di data test.

## Dataset

Datanya `Ames_Housing_Sales.csv`, penjualan rumah di Ames, Iowa (versi dari kompetisi Kaggle "House Prices"). Ada 79 kolom fitur dan satu kolom target, `SalePrice`. Dibaca langsung dari link, jadi tidak perlu diunduh:

https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML240EN-SkillsNetwork/labs/data/Ames_Housing_Sales.csv

## 1. Import library dan baca data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler, MinMaxScaler, MaxAbsScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

sns.set_style("whitegrid")

In [ ]:
URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML240EN-SkillsNetwork/labs/data/Ames_Housing_Sales.csv"
data = pd.read_csv(URL)
data.head()

### Latihan 1: ukuran data dan tipe kolom

In [ ]:
print("ukuran data:", data.shape)
data.dtypes.value_counts()

Ada tiga jenis tipe: integer, float, dan object (teks/kategorikal). Kolom object inilah yang nanti perlu di-encode sebelum masuk ke regresi linear.

## 2. Berapa kolom yang muncul kalau semua kategorikal di-one-hot encode?

One-hot encoding mengubah satu kolom kategorikal dengan `n` kategori menjadi `n-1` kolom baru (satu kategori dibuang supaya tidak redundan). Kolom yang isinya cuma satu nilai tidak perlu di-encode. Aku hitung dulu perkiraannya sebelum benar-benar di-encode.

In [ ]:
kolom_kat = data.select_dtypes(exclude="number").columns
jumlah_kategori = data[kolom_kat].nunique().sort_values(ascending=False)
jumlah_kategori.head(10)

In [ ]:
perlu_encode = jumlah_kategori[jumlah_kategori > 1]
kolom_baru = int((perlu_encode - 1).sum())

print("kolom kategorikal            :", len(kolom_kat))
print("kolom one-hot yang terbentuk :", kolom_baru)

Jumlah kolomnya bertambah cukup banyak untuk dataset yang barisnya cuma sekitar seribuan, ini yang nanti berpengaruh ke overfitting.

## 3. Buat dua versi data

- `data_ohe`: semua kolom kategorikal di-one-hot encode (pakai `pd.get_dummies`, kolom aslinya otomatis hilang)
- `data_num`: kolom kategorikal dibuang, hanya sisa kolom numerik

In [ ]:
data_ohe = pd.get_dummies(data, columns=list(kolom_kat), drop_first=True, dtype=int)
data_num = data.drop(columns=kolom_kat)

print("tanpa encoding :", data_num.shape)
print("one-hot        :", data_ohe.shape)
print("kolom tambahan :", data_ohe.shape[1] - data_num.shape[1], "| hitungan sebelumnya:", kolom_baru)

## 4. Train-test split dan perbandingan error

Kedua versi data dipisah dengan `random_state` yang sama supaya baris yang masuk train dan test identik.

In [ ]:
def pisah(df, target="SalePrice"):
    X = df.drop(columns=target)
    y = df[target]
    return train_test_split(X, y, test_size=0.3, random_state=42)

Xtr_num, Xte_num, ytr_num, yte_num = pisah(data_num)
Xtr_ohe, Xte_ohe, ytr_ohe, yte_ohe = pisah(data_ohe)

print("index train sama:", (Xtr_num.index == Xtr_ohe.index).all())
print("train:", len(Xtr_num), "| test:", len(Xte_num))

In [ ]:
def mse_train_test(Xa, ya, Xb, yb):
    model = LinearRegression().fit(Xa, ya)
    return {"train": mean_squared_error(ya, model.predict(Xa)),
            "test": mean_squared_error(yb, model.predict(Xb))}

hasil = pd.DataFrame({
    "tanpa encoding": mse_train_test(Xtr_num, ytr_num, Xte_num, yte_num),
    "one-hot": mse_train_test(Xtr_ohe, ytr_ohe, Xte_ohe, yte_ohe),
})
hasil.round(0)

Cara membaca tabelnya: model yang bagus punya error train dan test yang dekat. Kalau error train kecil tapi error test jauh lebih besar, modelnya overfitting (menghafal data latih). Model one-hot punya ratusan fitur tambahan dibanding jumlah data latih, jadi lebih rawan overfit. Masalah inilah yang nanti diatasi dengan regularisasi di materi berikutnya.

## 5. Scaling

Scaling hanya dikenakan pada kolom float (kolom hasil one-hot tidak perlu). Aturan pentingnya: scaler di-`fit` hanya pada data train, lalu dipakai untuk mengubah data train dan test. Supaya aturan itu terjaga otomatis, aku pakai `ColumnTransformer` di dalam `Pipeline`.

Cek dulu kemiringan (skewness) kolom float di data train:

In [ ]:
kolom_float = Xtr_num.select_dtypes("float").columns.tolist()
Xtr_num[kolom_float].skew().sort_values(ascending=False).head(8).round(2)

Beberapa kolom cukup miring, tapi di latihan ini aku fokus ke efek scaler saja dan tidak melakukan transformasi log.

In [ ]:
daftar_scaler = {"standard": StandardScaler, "minmax": MinMaxScaler, "maxabs": MaxAbsScaler}
daftar_data = {
    "tanpa encoding": (Xtr_num, ytr_num, Xte_num, yte_num),
    "one-hot": (Xtr_ohe, ytr_ohe, Xte_ohe, yte_ohe),
}

baris = []
for nama_data, (Xa, ya, Xb, yb) in daftar_data.items():
    hasil_scaler = {}
    for nama_sc, kelas in daftar_scaler.items():
        pipa = Pipeline([
            ("scale", ColumnTransformer([("float", kelas(), kolom_float)], remainder="passthrough")),
            ("lr", LinearRegression()),
        ])
        pipa.fit(Xa, ya)
        hasil_scaler[nama_sc] = mean_squared_error(yb, pipa.predict(Xb))
    baris.append(pd.Series(hasil_scaler, name=nama_data))

pd.DataFrame(baris).round(0)

Untuk regresi linear biasa, scaling hampir tidak mengubah prediksi karena model ini tidak sensitif terhadap skala fitur, jadi angka antar scaler seharusnya sangat dekat. Scaling baru berpengaruh besar saat ada regularisasi atau gradient descent (dibahas di notebook berikutnya).

## 6. Prediksi vs nilai sebenarnya

In [ ]:
lr = LinearRegression().fit(Xtr_num, ytr_num)
pred = lr.predict(Xte_num)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(yte_num, pred, s=15, alpha=0.5)
batas = [yte_num.min(), yte_num.max()]
ax.plot(batas, batas, "r--", label="prediksi sempurna")
ax.set(xlabel="Harga sebenarnya", ylabel="Harga prediksi",
       title="Regresi linear (tanpa encoding): prediksi vs aktual")
ax.legend()
plt.show()

Titik yang dekat garis putus-putus berarti prediksinya akurat. Makin jauh dari garis, makin besar kesalahannya.